In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.19 The Transverse-Field Ising Chain: A Phase Transition at Absolute Zero

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.19",
    title="The Transverse-Field Ising Chain: A Phase Transition at Absolute Zero",
    blurb="Every transition so far needed heat; this one happens where there is none. A "
    "chain of spins whose coupling and field refuse to commute must reorganize its "
    "ground state at a critical field — and the chain secretly solves itself: it is "
    "a gas of free fermions in disguise, its transition a gap closing. We check "
    "four thousand amplitudes against a one-line momentum sum at machine "
    "precision, watch symmetry breaking cast exponentially thin shadows in finite "
    "systems, and measure the central charge of a conformal field theory — one half, "
    "to half a percent — from fourteen spins.",
    difficulty="advanced",
    estimate="205–245 min",
)

## Notebook overview

Every transition in this course so far needed heat: order lost to thermal agitation as $T$
rose, the bargain between energy and entropy that Volume V made its engine. This notebook's transition happens at **absolute zero**, where there is no heat at all: the agitation is quantum. The Hamiltonian $H = -J\sum\sigma^z_i\sigma^z_{i+1} - h\sum\sigma^x_i$ contains
two demands that do not commute: the coupling wants definite $\sigma^z$ (aligned
neighbours), the field wants definite $\sigma^x$, and no state satisfies both. The ground state must broker a compromise, and at $h = J$ the compromise changes character abruptly.
The classical chain of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) famously has *no* transition at any finite temperature in one
dimension; its quantum counterpart has one at $T = 0$, with $g = h/J$ replacing temperature
as the dial.

The rendezvous reveals whose machinery this really is. The Jordan–Wigner transformation
(sketched honestly, not belabored) maps the chain to **free fermions** with the Bogoliubov
spectrum $\varepsilon(k) = 2\sqrt{J^2 + h^2 - 2Jh\cos k}$ (Pfeuty 1970), so the ground state is a *filled Fermi sea* of quasiparticles (Movement III hiding inside a magnet), and the ground energy is a one-line momentum sum over the antiperiodic set $k = (2j{-}1)\pi/N$.
Exact diagonalization agrees with that sum to $10^{-14}$ at nine $(N, h)$ points: four
thousand amplitudes against six terms. In fermion language the transition is transparent: the quasiparticle gap $\varepsilon(0) = 2|h - J|$ closes linearly at $h = J$, and a closing
gap *is* the transition ($z = 1$: energy scales as one over length). The gap then tells four
stories through one parity rule (quasiparticles are made in pairs within a fermion-parity
sector; a single quasiparticle changes sector): the ordered doublet splits *exponentially* ($\sim h^N$: the finite-size shadow of symmetry breaking, and the states are cats); the
ordered side's physical gap is a *two*-quasiparticle threshold (a flipped domain has two
walls); the disordered gap is a single quasiparticle at $2(h-J)$; and the critical gap closes as $\Delta \cdot N \to \pi/2$, with an honest correction performed in public, since
the naive same-sector guess predicts a different coefficient and the computer overruled it.

Order turns out to be subtler than magnetization: any finite symmetric ground state has
$\langle\sigma^z\rangle = 0$ *identically* (measured: $10^{-11}$, which is solver noise, not
physics), and the order lives in correlations: $C(N/2) = 0.93054$ against Pfeuty's exact $(1-h^2)^{1/4} = 0.93060$. The exponent inside, $1/8$, is the 2D *classical* Ising
magnetization exponent: a quantum chain at $T = 0$ is secretly a two-dimensional classical
model, with imaginary time as the second dimension: the deepest flag this volume plants, with the mechanism deferred wholesale to [§7.20](imaginary-time-quantum-classical.ipynb) and Onsager waiting there. The showpiece is
entanglement: the half-chain entropy obeys an area law off criticality (saturating to four
digits), carries a teachable gem in the ordered phase ($S \approx \ln 2$: *cat* entropy, not
correlation entropy), and grows as $(c/3)\ln N$ at the critical point; the fit over $N = 8$–$14$ returns $c = 0.497$ against the Ising CFT's $c = 1/2$: a conformal field
theory's central charge, measured to half a percent from fourteen spins on a laptop.
Temperature returns at the close: the full $N = 10$ spectrum paints $C(T, h)$, and the **quantum-critical fan** opens upward from $(h = J, T = 0)$: each gapped phase suppressed by its own $e^{-\Delta/T}$ while the critical column, feeling no gap, keeps $C \propto T$.
A zero-temperature critical point, organizing finite-temperature physics above it.

> **Conventions (this notebook).** $J = 1$ (all energies in units of the coupling);
> $g = h/J$ is the dial; periodic boundary conditions; $N$ even throughout. Sparse Pauli
> strings via `scipy.sparse.kron` chains; low states from `scipy.sparse.linalg.eigsh`
> (`which='SA'`, `k` stated per use, tolerance at machine default — the ordered doublet's
> $10^{-5}$ splitting needs $k \ge 4$ to resolve both members reliably); the dense
> `numpy.linalg.eigvalsh` appears exactly once, for the $N = 10$ full-spectrum
> thermodynamics. Memory honesty: a state costs $2^N$ complex amplitudes ($N = 14$:
> 16384 — fine), a dense matrix $4^N$ ($N = 14$: 2.1 GB — never formed; the
> sparse-before-dense rule). The Jordan–Wigner sum runs over the antiperiodic momenta
> $k = (2j{-}1)\pi/N$ (the even-fermion-parity sector, where the finite-$N$ ground state
> lives). Schmidt spectra pass a $p > 10^{-14}$ floor *before* the logarithm; the central
> charge fit uses even $N$ only; thermal sums subtract $E_0$ before exponentiating (the
> discipline of [§7.4](thermal-density-matrix.ipynb)).
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the two limiting phases by direct computation; ED against the
> Jordan–Wigner sum at $10^{-12}$; the four gap behaviours against their quasiparticle
> predictions; Pfeuty's correlation at four digits; the area-law saturation and the
> $c = 1/2$ fit; the fan's three columns against their own gaps. A ✓ is strong evidence; a
> ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** The quantum–classical mapping is flagged once and deferred wholesale to [§7.20](imaginary-time-quantum-classical.ipynb);
> DMRG and tensor networks (whose existence the area law explains), Kibble–Zurek dynamics,
> and the experimental chains (CoNb₂O₆ neutron spectroscopy, Rydberg arrays, trapped ions)
> are named horizons. See Pfeuty, Ann. Phys. **57**, 79 (1970); Sachdev, *Quantum Phase
> Transitions*; Calabrese & Cardy 2004; Kogut 1979 (for [§7.20](imaginary-time-quantum-classical.ipynb)). Cross-reference [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (the
> classical chain and the finite-size-scaling training), [§7.7](bose-einstein-fermi-dirac.ipynb)/[§7.9](fermi-gas-zero-temperature.ipynb) (the Fermi sea inside the
> magnet), [§7.18](quantum-paramagnets-brillouin.ipynb) (independence's end), [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)/[§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the algebra), [§7.4](thermal-density-matrix.ipynb) (log-sum-exp, again),
> and forward to [§7.20](imaginary-time-quantum-classical.ipynb)/[§7.21](path-integral-monte-carlo.ipynb).

## Theory in brief

### The model, and why it must have a transition

The chain is that of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb), translated into operators: the same nearest-neighbour coupling, now
written in Pauli matrices, plus the one ingredient a classical chain cannot supply — a
transverse field whose $\sigma^x$ refuses to commute with the coupling's $\sigma^z$'s. The
model, and the non-commutation that powers everything below:

```{math}
:label: eq-tfim
H = -J\sum_{i=0}^{N-1}\sigma^z_i\sigma^z_{i+1} \;-\; h\sum_{i=0}^{N-1}\sigma^x_i,
\qquad
\big[\sigma^z_i\sigma^z_{i+1},\ \sigma^x_i\big] \neq 0 .
```

The coupling is minimized by states of definite $\sigma^z$ (the two ferromagnets), the field by the definite-$\sigma^x$ product state. Because the two terms do not commute, no
state satisfies both, and the ground state is a compromise whose character depends on
$g = h/J$. The two certainties (verified by exact diagonalization below): at $h = 0$, a
doubly degenerate ferromagnet with $\langle\sigma^z_0\sigma^z_r\rangle = 1$ at every $r$;
at $h \gg J$, a unique paramagnet polarized along $x$. Between them something must give.
The engine deserves three careful sentences against Volume V's: a thermal transition is a
bargain between energy and entropy, struck at $T_c$, and entropy needs temperature to have
a seat at the table. Here $T = 0$ and entropy never arrives; the competition is *internal*: two non-commuting pieces of one Hamiltonian, with quantum fluctuations (the field's
$\sigma^z$-flipping) playing the role thermal fluctuations play classically. The parallel
is real but not an identity: what closes at the critical point is an energy gap, not a
free-energy barrier, and the dial is $g$, not $T$.

### Sparse construction

To diagonalize anything we first need $H$ as a matrix: $N$ spins live in the tensor product
of $N$ two-dimensional spaces ([§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)/[§6.20](../06-quantum-mechanics/identical-particles.ipynb) built this algebra), so a single-site Pauli
operator acts on the chain as a Kronecker chain with $\sigma^\alpha$ in slot $i$ and
identities everywhere else:

```{math}
:label: eq-sparse-ed
\sigma^\alpha_i = \mathbb{1}_2 \otimes \cdots \otimes \sigma^\alpha \otimes \cdots \otimes \mathbb{1}_2
\quad(\text{site } i \text{ of } N),
\qquad
\dim H = 2^N .
```

Pauli strings are `scipy.sparse.kron` chains (the `op_at` helper); $H$ assembles term by
term into CSR. Low eigenpairs come from `eigsh(k, which='SA')`; Lanczos never forms the dense matrix, which is what makes $N = 14$ (dimension 16384) routine. The memory wall in
numbers: dense $N = 14$ would be $16384^2$ doubles $= 2.1$ GB and $N = 16$ would be 34 GB;
sparse $H$ at $N = 14$ holds $\sim N \cdot 2^N$ entries — a few megabytes.

### The rendezvous: the chain is free fermions

The chain solves itself: the Jordan–Wigner transformation trades spins for fermions, the
Hamiltonian comes out quadratic, and a Bogoliubov rotation diagonalizes it outright. We
sketch the map below and refer the full derivation to Pfeuty 1970 (Sachdev's *Quantum Phase
Transitions* gives the modern account); what survives into the numerics is a spectrum and a
one-line sum:

```{math}
:label: eq-jordan-wigner
\varepsilon(k) = 2\sqrt{J^2 + h^2 - 2Jh\cos k},
\qquad
E_0 = -\tfrac12\sum_{k}\varepsilon(k),
\quad k = \frac{(2j-1)\pi}{N},\ j = 1,\dots,N .
```

Jordan–Wigner in half a page: map spin-down/spin-up at site $i$ to fermion
occupation $0/1$, with a *string* of $\sigma^z$'s attached to each fermion operator to repair the commutation algebra (spins on different sites commute; fermions anticommute; the string supplies the minus signs). Under the map, the transverse field counts fermions
and the Ising coupling hops and pairs them: a quadratic, *free*, fermion Hamiltonian, diagonalized by a Bogoliubov rotation into quasiparticles with the spectrum above (Pfeuty 1970). The physical reading: a quasiparticle is a dressed **domain wall** on the ordered
side and a dressed **spin flip** on the disordered side. The finite-$N$ ground state lives
in the even fermion-parity sector, whose momenta are antiperiodic — the $k$-set in
{eq}`eq-jordan-wigner` — and its energy is the filled quasiparticle sea: Movement III's oldest move, inside a magnet. The transition, in these coordinates, is transparent: $\varepsilon(k)$ is minimized at $k = 0$ with $\varepsilon(0) = 2|h - J|$; the gap closes linearly at $h = J$, and a closing gap is the transition ($z = 1$: the energy scale
vanishes like one over the length scale).

### The gap's four stories: parity bookkeeping

With the spectrum known, the low-lying gaps become a counting exercise: quasiparticles cost
$\varepsilon(k)$ apiece, and the conserved fermion parity dictates whether one or two can be
made at a time. Four regimes, four scalings, stated first and unpacked after:

```{math}
:label: eq-gap-stories
\Delta_{\text{doublet}} \sim h^N,
\qquad
\Delta_{\text{ordered}} \to 2\,\varepsilon(0),
\qquad
\Delta_{\text{disordered}} = \varepsilon(0) = 2(h{-}J),
\qquad
\Delta_{\text{critical}} \cdot N \to \frac{\pi}{2}.
```

One rule organizes everything: Jordan–Wigner fermion parity is conserved, quasiparticles
are created in *pairs* within a sector, and the lowest state of the *other* sector differs
by a *single* quasiparticle. (i) Ordered side, cross-sector: the two ferromagnetic ground states survive at finite $N$ as a doublet split exponentially, $\sim h^N$ (the amplitude to tunnel all $N$ spins), and the two eigenstates are **cats**,
$(|\!\uparrow\cdots\uparrow\rangle \pm |\!\downarrow\cdots\downarrow\rangle)/\sqrt2$; in
the thermodynamic limit the splitting vanishes and any perturbation selects a broken
ferromagnet. (ii) Ordered side, same-sector: the physical excitation is *two* quasiparticles (a flipped domain has two walls), so the gap approaches
$2\varepsilon(0) = 4(J-h)$. (iii) Disordered side: a single spin flip changes parity, one
quasiparticle suffices, and the gap is $\varepsilon(0) = 2(h-J)$ exactly. (iv) Critical: the cross-sector gap closes as $(\pi/2)/N$, and the honest note is that the coefficient is a *cross-sector* fact: the naive same-sector estimate $2\varepsilon(\pi/N)$ predicts a
different number, and the computation corrects it in public below.

### Order without magnetization

A subtlety guards the ordered phase: its obvious order parameter can be forced to vanish by
symmetry alone, so the honest question is what survives at long distance in the two-point
function. Pfeuty 1970 answers in closed form:

```{math}
:label: eq-order-correlations
\langle\sigma^z\rangle_{\text{finite, symmetric}} = 0
\qquad\text{but}\qquad
C(r) = \langle\sigma^z_0\sigma^z_r\rangle \xrightarrow{r\to\infty} m^2
= \left(1 - (h/J)^2\right)^{1/4}.
```

Any finite symmetric eigenstate magnetizes *exactly nothing* ($\sigma^z \to -\sigma^z$ is a symmetry of $H$, and the cat eigenstates respect it), so a measured
$\langle\sigma^z\rangle \sim 10^{-11}$ is Lanczos noise, taught as symmetry rather than
smallness. The order hides in correlations: $C(r)$ saturates to $m^2$ with Pfeuty's exact
$m = (1 - g^2)^{1/8}$. Read the exponent aloud: $1/8$ is the magnetization exponent
$\beta$ of the *two-dimensional classical* Ising model — because a 1D quantum chain at
$T = 0$ *is* a 2D classical model, with imaginary time as the second dimension. The flag is
planted here and the mechanism deferred wholesale to [§7.20](imaginary-time-quantum-classical.ipynb); Onsager is the appointment.

### Entanglement: the central charge, measured

An entanglement entropy attaches to any bipartition of the ground state, and its growth
with $N$ is the sharpest diagnostic the notebook owns. The three behaviours below are
quoted rather than derived: the critical law, with its universal $c/3$ prefactor, is a
theorem of conformal field theory (Calabrese & Cardy 2004):

```{math}
:label: eq-tfi-entanglement
S = -\sum_i p_i\ln p_i,
\qquad
S_{\text{off-critical}} \to \text{const (area law)},
\qquad
S_{\text{critical}} = \frac{c}{3}\ln N + \text{const},\ \ c = \tfrac12 .
```

Cut the chain in half, Schmidt-decompose the ground state (`reshape` to
$2^{N/2} \times 2^{N/2}$ plus `numpy.linalg.svd`), and sum $-p\ln p$ over the squared
singular values. Off criticality the entropy *saturates* with $N$: the area law, and the reason DMRG exists. The ordered phase hides a gem: its $S \approx \ln 2$ is **cat entropy** (tracing half of $(|\!\uparrow\cdots\rangle + |\!\downarrow\cdots\rangle)/\sqrt2$ leaves a two-outcome classical mixture worth exactly $\ln 2$), not correlation entropy,
which is why $S(h)$ barely peaks over the ordered side at small $N$ and why the honest
critical signature is the *scaling*: at $h = J$ conformal invariance makes the entropy grow
logarithmically with the famous $c/3$ coefficient (Holzhey–Wilczek; Calabrese–Cardy), and
fitting $S$ against $\ln N$ over $N = 8$–$14$ measures the Ising CFT's central charge.

### The quantum-critical fan

One dial remains: with every eigenvalue of a small chain in hand, the canonical ensemble is
exact, and the heat capacity follows as an energy variance (differentiate $\langle E\rangle$
once in $T$ and the fluctuation formula appears). Each phase's gap then dictates its own
low-$T$ law:

```{math}
:label: eq-qc-fan
C(T, h) = \frac{\mathrm{Var}(E)}{NT^2}
\qquad\text{with}\qquad
C \sim e^{-\Delta/T} \ \text{(gapped columns)},
\quad
C \propto T \ \text{(the critical column)}.
```

Temperature returns through the full $N = 10$ spectrum (dense `eigvalsh`, 1024 states: the one dense diagonalization the notebook permits itself). At low $T$ each gapped phase is
suppressed by *its own* gap, while the critical column feels none and its heat capacity is linear in $T$: the $c = \tfrac12$ CFT's law, valid down to the finite-size floor
$\Delta \sim \pi/2N$. The picture is Sachdev's quantum-critical fan: a $T = 0$ critical point governing finite-temperature physics in a wedge that *widens* upward: criticality you can feel at temperatures far above absolute zero.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: J = 1 throughout (g = h); periodic boundaries; N even. Sparse
# before dense, always: a dense H at N = 14 would be 2.1 GB, while the CSR H
# holds a few MB. The one dense eigvalsh in the notebook is the N = 10 full
# spectrum (1024 states) for the thermodynamic fan.
SX = sp.csr_matrix(np.array([[0.0, 1.0], [1.0, 0.0]]))
SZ = sp.csr_matrix(np.array([[1.0, 0.0], [0.0, -1.0]]))
ID2 = sp.identity(2, format="csr")


def op_at(o, i, N):
    """The single-site operator o acting at site i of an N-site chain, as a sparse kron chain.

    Builds 1 ⊗ … ⊗ o ⊗ … ⊗ 1 with scipy.sparse.kron, site 0 leftmost — the
    ordering that makes a half-chain Schmidt cut a plain reshape (sites 0..N/2−1
    become the row index). CSR output; never densified.

    Parameters
    ----------
    o : scipy.sparse matrix
        The 2×2 single-site operator.
    i : int
        Site index, 0-based.
    N : int
        Chain length.

    Returns
    -------
    scipy.sparse.csr_matrix
        The 2^N × 2^N operator.
    """
    out = sp.identity(1, format="csr")
    for j in range(N):
        out = sp.kron(out, o if j == i else ID2, format="csr")
    return out


def H_tfim(N, h, J=1.0):
    """The transverse-field Ising Hamiltonian, sparse: −J Σ sz_i sz_{i+1} − h Σ sx_i (PBC).

    Assembled term by term from op_at strings (eq-tfim / eq-sparse-ed). The
    periodic bond (N−1, 0) is included; J = 1 sets the energy unit.

    Parameters
    ----------
    N : int
        Chain length (even).
    h : float
        Transverse field.
    J : float, optional
        Ising coupling (default 1: the unit).

    Returns
    -------
    scipy.sparse.csr_matrix
        H, 2^N × 2^N.
    """
    H = sp.csr_matrix((2**N, 2**N))
    for i in range(N):
        H = H - J * (op_at(SZ, i, N) @ op_at(SZ, (i + 1) % N, N)) - h * op_at(SX, i, N)
    return H


def low_spectrum(N, h, k=4):
    """The k lowest eigenpairs of the chain by scipy.sparse.linalg.eigsh (which='SA').

    Lanczos on the CSR Hamiltonian; tolerance left at the machine default —
    needed, because the ordered phase's ground doublet is split by only ~1e-5
    at N = 12 and a loose solve can miss one member and silently mislabel every
    gap above it (the near-degeneracy trap). k ≥ 4 wherever the doublet matters.

    Parameters
    ----------
    N : int
        Chain length.
    h : float
        Transverse field.
    k : int, optional
        Number of eigenpairs (default 4).

    Returns
    -------
    tuple
        (evals sorted ascending, evecs column-matched).
    """
    vals, vecs = eigsh(H_tfim(N, h), k=k, which="SA")
    order = np.argsort(vals)
    return vals[order], vecs[:, order]


def jw_energy(N, h, J=1.0):
    """The exact ground energy by the Jordan–Wigner/Pfeuty momentum sum (eq-jordan-wigner).

    E_0 = −(1/2) Σ ε(k) with ε(k) = 2√(J^2 + h^2 − 2Jh cos k) over the
    ANTIPERIODIC set k = (2j−1)π/N, j = 1..N — the even-fermion-parity sector,
    where the finite-N ground state lives. Six terms at N = 12 do the work of a
    4096-dimensional diagonalization; Exercise 2 gates the agreement at 1e-12.

    Parameters
    ----------
    N : int
        Chain length.
    h : float
        Transverse field.
    J : float, optional
        Coupling (default 1).

    Returns
    -------
    float
        The ground-state energy.
    """
    k = (2.0 * np.arange(1, N + 1) - 1.0) * np.pi / N
    eps = 2.0 * np.sqrt(J**2 + h**2 - 2.0 * J * h * np.cos(k))
    return -0.5 * float(np.sum(eps))


def eps_k(k, h, J=1.0):
    """Pfeuty's quasiparticle dispersion ε(k) = 2√(J^2 + h^2 − 2Jh cos k).

    The Bogoliubov spectrum of the free-fermion chain: minimum ε(0) = 2|h − J|
    (the gap that closes at the transition), maximum ε(π) = 2(h + J).

    Parameters
    ----------
    k : float or numpy.ndarray
        Momentum.
    h : float
        Transverse field.
    J : float, optional
        Coupling (default 1).

    Returns
    -------
    float or numpy.ndarray
        ε(k).
    """
    return 2.0 * np.sqrt(J**2 + h**2 - 2.0 * J * h * np.cos(np.asarray(k, dtype=float)))


def correlation(psi, r, N):
    """The order correlator C(r) = ⟨ψ| sz_0 sz_r |ψ⟩ (eq-order-correlations).

    Sparse matrix–vector products only; the operator is diagonal, so the cost is
    one elementwise pass per call.

    Parameters
    ----------
    psi : numpy.ndarray
        State vector, length 2^N.
    r : int
        Separation.
    N : int
        Chain length.

    Returns
    -------
    float
        C(r).
    """
    op = op_at(SZ, 0, N) @ op_at(SZ, r % N, N)
    return float(psi @ (op @ psi))


def half_chain_entropy(psi, N):
    """The half-chain entanglement entropy by Schmidt decomposition (eq-entanglement).

    reshape(2^{N/2}, 2^{N/2}) — valid because op_at's kron order puts sites
    0..N/2−1 in the row index — then numpy.linalg.svd for the Schmidt spectrum
    p_i = s_i^2, and S = −Σ p ln p over p > 1e-14 ONLY: smaller weights are
    numerical dust whose log is NaN-bait (the stated floor, applied BEFORE the
    logarithm).

    Parameters
    ----------
    psi : numpy.ndarray
        State vector, length 2^N (N even).
    N : int
        Chain length.

    Returns
    -------
    float
        S in nats.
    """
    half = 2 ** (N // 2)
    s = np.linalg.svd(psi.reshape(half, half), compute_uv=False)
    p = s**2
    p = p[p > 1e-14]
    return float(-np.sum(p * np.log(p)))


def thermal_C(evals, T, N):
    """Heat capacity per site from a full spectrum: C = Var(E)/(N T^2) (eq-qc-fan).

    E_0 is subtracted before exponentiating — the log-sum-exp discipline of §7.4 in
    its simplest form: at T = 0.1 the raw e^{−E/T} underflows for every state
    of an N = 10 chain (E_0 ≈ −12), while the shifted weights start at 1.

    Parameters
    ----------
    evals : numpy.ndarray
        The full eigenvalue list.
    T : float
        Temperature.
    N : int
        Chain length (for the per-site normalization).

    Returns
    -------
    float
        C/(N k_B).
    """
    E = evals - evals.min()
    w = np.exp(-E / T)
    Z = np.sum(w)
    E_avg = np.sum(E * w) / Z
    E2_avg = np.sum(E**2 * w) / Z
    return float((E2_avg - E_avg**2) / (N * T**2))


print(
    f"memory honesty: dense H at N = 14 would be {(2**14)**2 * 8 / 1e9:.1f} GB — never formed;"
)
print(
    f"sparse H at N = 14: ~{14 * 2**14 * 12 / 1e6:.0f} MB of CSR. States are fine: 2^14 = {2**14} amplitudes."
)

## Exercise 1 — Two demands that cannot both be met

The model built sparse, and its two limiting certainties verified. Cite {eq}`eq-tfim`,
{eq}`eq-sparse-ed`.

1. Implement `op_at` (sparse `scipy.sparse.kron` chains) and `H_tfim`; state the memory
   budget ($2^N$ with numbers; the sparse-before-dense rule).
2. Verify the $h = 0$ limit by `eigsh`: two degenerate ground states with
   $\langle\sigma^z_0\sigma^z_r\rangle = 1$ at every $r$.
3. Verify the $h \gg J$ limit ($h = 8$): a unique ground state with
   $\langle\sigma^x\rangle \to 1$ and $C(r) \to 0$.
4. State the engine (prose): $[\text{coupling}, \text{field}] \neq 0$, so no state satisfies both; $g = h/J$ replaces temperature as the dial, and something must give at a critical
   $g$ (Volume V's engine contrasted in three careful sentences).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    deg_split < 1e-10 and max(abs(c - 1.0) for c in C_ferro) < 1e-9,
    "the h = 0 certainty: a degenerate ferromagnetic doublet with perfect σz correlation",
    f"splitting {deg_split:.1e}",
)
validate.check(
    gap_para > 1.0 and sx_avg > 0.995 and abs(C_para) < 0.01,
    "the h >> J certainty: a unique x-polarized paramagnet with no σz order",
    f"⟨σx⟩ = {sx_avg:.4f}",
)

## Exercise 2 — The rendezvous: four thousand amplitudes vs one momentum sum

The chain is free fermions, and two unrelated computations agree to machine precision.
Cite {eq}`eq-jordan-wigner`.

1. Sketch Jordan–Wigner in half a page (spins ↔ fermions; the string; the domain-wall
   reading) and state Pfeuty's $\varepsilon(k)$ with the antiperiodic $k$-set.
2. Implement `jw_energy` and verify against `eigsh` ground energies at
   $N = 8, 10, 12 \times h = 0.5, 1.0, 1.5$: nine agreements at $\le 10^{-12}$.
3. Read the physics in fermion language (prose + one line): the ground state is a filled Bogoliubov sea (Movement III inside a magnet); the gap is $\varepsilon(0) = 2|h - J|$; the transition is a Fermi-sea gap closing.
4. Reflect on the rendezvous (prose): a 16384-dimensional diagonalization and a six-term
   momentum sum had no obligation to agree; they agree because Jordan and Wigner found the right coordinates: the course's recurring lesson about representations.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    ED_grid,
    JW_grid,
    "the chain is free fermions: nine agreements at machine precision",
    atol=1e-12,
)

## Exercise 3 — The gap's four stories

One parity rule, four verified behaviours, including a correction performed in public.
Cite {eq}`eq-gap-stories`.

1. State the parity bookkeeping (pairs within a sector, singles across) and compute
   $E_0\dots E_3$ via `eigsh(k=4, which='SA')` (machine tolerance; the near-degeneracy trap: a loose solve misses a doublet member and mislabels every gap).
2. Verify the ordered side at $h = 0.5$: doublet splitting $\sim h^N$ across $N = 8, 10,
   12$ (cat states named), and the physical two-quasiparticle gap $E_2 - E_0$ against the
   quasiparticle prediction $2\varepsilon(\pi/N) \to 2\varepsilon(0) = 2$.
3. Verify the disordered side ($\Delta = 2(h-J)$ at $h = 1.5$) and the critical closing
   $\Delta\cdot N \to \pi/2$ across $N = 8$–$14$.
4. Perform the correction in public (prose): the naive same-sector estimate $2\varepsilon(\pi/N)$ predicts $\Delta\cdot N \to 4\pi \ne \pi/2$; the measured coefficient is a cross-sector fact; $z = 1$ confirmed, sector bookkeeping the lesson,
   and the course's rule restated: predictions are drafts until the computer countersigns.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [split[8], split[10], split[12]],
    [1.5e-3, 3.2e-4, 7.2e-5],
    "the cat doublet: exponential splitting ~h^N, the finite-size shadow of symmetry breaking",
    rtol=3e-1,
)
validate.close(
    [gap2[8], gap2[10], gap2[12]],
    [
        2 * eps_k(np.pi / 8, h_ord),
        2 * eps_k(np.pi / 10, h_ord),
        2 * eps_k(np.pi / 12, h_ord),
    ],
    "the ordered physical gap is two quasiparticles: two domain walls, priced by Pfeuty",
    rtol=1e-2,
)
validate.close(
    [gap_dis, gapN[14]],
    [1.003, 1.572],
    "one quasiparticle on the disordered side; Δ·N → π/2 at criticality (z = 1)",
    rtol=5e-3,
)
validate.check(
    gapN[8] > gapN[10] > gapN[12] > gapN[14] > np.pi / 2,
    "and the critical product descends monotonically onto π/2 — the cross-sector coefficient",
    f"Δ·N: {gapN[8]:.4f} → {gapN[14]:.4f}",
)

## Exercise 4 — Order without magnetization

The symmetric ground state hides its order in correlations, and the exponent it carries belongs to another dimension. Cite {eq}`eq-order-correlations`.

1. Measure $\langle\sigma^z\rangle$ at $h = 0.5$, $N = 12$ and teach it as *exact* symmetry (any residue is Lanczos noise; the $\le 10^{-8}$ audit stated), with the
   thermodynamic-limit symmetry-breaking story in three sentences.
2. Compute $C(r) = \langle\sigma^z_0\sigma^z_r\rangle$ across the chain via `correlation`;
   verify $C(N/2)$ against Pfeuty's $(1 - h^2)^{1/4}$ to four digits, and the disordered
   decay at $h = 1.5$.
3. Plot $C(N/2)$ vs $h$ across the transition for $N = 8, 10, 12$: the order parameter
   emerging with system size.
4. Read the exponent aloud (prose): $m = (1 - g^2)^{1/8}$, and $1/8$ is the 2D *classical* Ising $\beta$; the flag planted for [§7.20](imaginary-time-quantum-classical.ipynb) (one dimension of imaginary time;
   Onsager the appointment) with no further leakage.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    abs(mz) < 1e-8,
    "⟨σz⟩ = 0 identically in the finite symmetric ground state — the residue is solver noise",
    f"measured {mz:.1e}",
)
validate.close(
    C_half,
    pfeuty,
    "order in correlations: Pfeuty to four digits",
    rtol=1e-3,
)
validate.check(
    C_half_dis < 0.05,
    "and the disordered side decays: no long-range order past the transition",
    f"C(6) = {C_half_dis:.3f}",
)

## Exercise 5 — The central charge, measured

Entanglement entropy: an area law, a cat's $\ln 2$, and a conformal number from fourteen
spins. Cite {eq}`eq-tfi-entanglement`.

1. Implement `half_chain_entropy` (`reshape` to $2^{N/2}\times2^{N/2}$ + `numpy.linalg.svd`; the $p > 10^{-14}$ floor *before* the log, the NaN trap) and compute $S(h)$ at $N = 12$.
2. Verify the area law off criticality: $S$ saturates to four digits between $N = 12$ and
   $N = 14$ at $h = 0.5$.
3. Teach the cat's share (prose + the numbers): ordered-phase $S \approx \ln 2 + \epsilon$, because tracing half of $(|\!\uparrow\cdots\rangle \pm |\!\downarrow\cdots\rangle)/\sqrt2$ leaves a two-outcome mixture; hence the subtle $S(h)$ peak at small $N$, and the honest critical signature is scaling, not the peak.
4. Measure $c$: fit $S(N/2)$ vs $\ln N$ at $h = 1$ over $N = 8, 10, 12, 14$
   (`numpy.polyfit`; even $N$ only): slope $= c/3$ against the Ising CFT's $c = 1/2$
   (Calabrese–Cardy credited; the sentence earned: a field theory's central charge, from a
   laptop).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    abs(S_vals[0.5] - S_14_ordered) < 5e-4,
    "the area law: off-critical entanglement saturates to four digits between N = 12 and 14",
    f"{S_vals[0.5]:.4f} vs {S_14_ordered:.4f}",
)
validate.check(
    abs(S_vals[0.5] - np.log(2)) < 0.02,
    "the cat's share: the ordered plateau is ln 2 of superposition bookkeeping, plus a small ε",
    f"S = {S_vals[0.5]:.4f}, ln 2 = {np.log(2):.4f}",
)
validate.close(
    c_measured,
    0.5,
    "c = 1/2, measured from fourteen spins",
    rtol=2e-2,
)

## Exercise 6 — The susceptibility sharpens

$\partial^2E_0/\partial h^2$ across sizes: the finite-size training of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb), run at $T = 0$.
Cite {eq}`eq-gap-stories`, {eq}`eq-order-correlations`.

1. Compute $\chi = -\partial^2(E_0/N)/\partial h^2$ by central differences (step
   $\delta h = 0.02$) across $h$ for $N = 8, 10, 12$, ground energies from `eigsh(k=1)`.
2. Verify the peak grows and sharpens toward $h = 1$ with $N$, and locate the finite-size
   pseudo-critical $h^*(N)$ drifting toward 1.
3. Overlay the Jordan–Wigner closed form's second derivative (the `jw_energy` sum,
   differenced with the same stencil) as the exact check.
4. Connect (prose): [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) trained this reading at finite $T$; here $N$ plays $T$'s role near a $T = 0$ transition, with finite-size scaling as the one language both transitions speak.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    peak_vals[8] < peak_vals[10] < peak_vals[12]
    and abs(peaks[12] - 1.0) <= abs(peaks[8] - 1.0) + 1e-9
    and jw_gap < 1e-8,
    "the transition announcing itself: the χ peak sharpens and drifts toward h = 1, on the JW curve",
    f"h*(N): {peaks[8]:.3f} → {peaks[12]:.3f}; peaks {peak_vals[8]:.3f} → {peak_vals[12]:.3f}",
)

## Exercise 7 — The quantum-critical fan

Temperature returns, and the $T = 0$ point governs a finite-$T$ wedge. Cite
{eq}`eq-qc-fan`.

1. Compute the full spectrum at $N = 10$ (dense `numpy.linalg.eigvalsh` on the 1024-state
   matrix — the notebook's one permitted densification) and implement `thermal_C` with
   $E_0$ subtracted before exponentiating (the log-sum-exp discipline of [§7.4](thermal-density-matrix.ipynb)).
2. Verify the low-$T$ columns at $T = 0.2$: $C/N$ at $h = 0.5$ (ordered, $\Delta \approx
   2$), $h = 1.0$ (critical), $h = 1.5$ (disordered, $\Delta = 1$): each gapped phase suppressed by its own $e^{-\Delta/T}$.
3. Map $C(T, h)$ and draw the fan: the V opening from $(h = 1, T = 0)$, with the
   finite-size floor $\Delta = \pi/2N$ bounding the critical column's gapless window.
4. Read the fan (prose): at criticality $C \propto T$ (the $c = \tfrac12$ CFT's linear law, window stated honestly): a zero-temperature critical point organizing finite-temperature physics above it (Sachdev named; CoNb₂O₆, Rydberg arrays, trapped
   ions in one outward breath).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [C_cols[0.5], C_cols[1.0], C_cols[1.5]],
    [0.0003, 0.024, 0.018],
    "the fan: each phase suppressed by its own gap, the critical column by none",
    rtol=2e-1,
)

## Exercise 8 — A transition without temperature

Volume V taught that phase transitions are bargains between energy and entropy, struck at
a temperature; this notebook found a transition where entropy never came to the table. Two terms of one Hamiltonian, unable to commute, forced the ground state itself to choose sides at $h = J$, and the choosing left every fingerprint the classical theory taught us
to look for: an order parameter (hidden in correlations, since finite symmetric systems
magnetize exactly nothing), a closing gap in place of a diverging correlation time, a
susceptibility peak sharpening with size, and critical exponents, one of which, the $1/8$, arrived wearing another model's clothes: the 2D classical Ising's, solved by
Onsager two decades before Pfeuty, one dimension up. That is not coincidence; it is the
deepest flag this volume has planted, and the next notebook takes it up: $\beta$ is a
length of imaginary time, and a quantum chain at $T = 0$ is a classical film strip ([§7.20](imaginary-time-quantum-classical.ipynb)).
Meanwhile the chain solved itself twice, by brute force and by becoming free fermions, and turned the redundancy into a measurement few courses attempt: the central charge of
a conformal field theory, one half to half a percent, extracted from fourteen spins.

There is something clarifying about a transition with no heat in it. Nothing jiggles,
nothing averages; the ground state simply cannot be two things at once, and at the
critical field it stops trying. Quantum mechanics does not need temperature to be
statistical — superposition was always an ensemble of a kind, and at $h = J$ the ensemble
reorganizes.

## Notebook summary

Movement V's summit: a phase transition at absolute zero, solved twice and measured.

- **The model** {eq}`eq-tfim`, {eq}`eq-sparse-ed`: non-commuting coupling and field; both limits verified by sparse ED (degenerate ferromagnet with $C(r) = 1$; unique $x$-paramagnet; both gated); $g = h/J$ as the dial that replaces temperature.
- **The rendezvous** {eq}`eq-jordan-wigner`: Jordan–Wigner makes the chain free fermions
  with Pfeuty's $\varepsilon(k)$; ED and the antiperiodic momentum sum agree at nine $(N, h)$ points to $10^{-14}$ (gated at $10^{-12}$): Movement III's Fermi sea inside a magnet, and the transition a gap closing at $\varepsilon(0) = 2|h-J|$.
- **Four gap stories** {eq}`eq-gap-stories`: the cat doublet's exponential splitting
  (gated), the two-quasiparticle ordered gap on Pfeuty's threshold (gated against
  $2\varepsilon(\pi/N)$), the one-quasiparticle disordered gap at $2(h-J)$ (gated), and the critical $\Delta\cdot N \to \pi/2$ (gated, monotone), with the naive same-sector coefficient corrected in public; $z = 1$.
- **Order without magnetization** {eq}`eq-order-correlations`:
  $\langle\sigma^z\rangle = 0$ exactly (gated as solver noise $\le 10^{-8}$); $C(N/2)$ on
  Pfeuty's $(1-h^2)^{1/4}$ to four digits (gated); the $1/8$ read aloud as 2D classical Ising's $\beta$: one flag for [§7.20](imaginary-time-quantum-classical.ipynb), no leakage.
- **The central charge** {eq}`eq-tfi-entanglement`: area-law saturation to four digits
  (gated); the ordered plateau as $\ln 2$ cat entropy (gated); and the critical $(c/3)\ln N$ fit returning $c = 0.497$ against the Ising CFT's $\tfrac12$ (gated at 2%), measured from fourteen spins.
- **The susceptibility**: the $T = 0$ response peak sharpening and drifting to $h = 1$ with $N$, on the Jordan–Wigner curve (gated); the finite-size grammar of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) at absolute zero.
- **The fan** {eq}`eq-qc-fan`: the $N = 10$ full spectrum paints $C(T, h)$; at $T = 0.2$
  the three columns sit at $0.0003/0.024/0.018$ (gated): each gapped phase throttled by its own $e^{-\Delta/T}$, the critical column linear in $T$ above the $\pi/2N$ floor:
  Sachdev's wedge.

Next door: why the $1/8$ belonged to a 2D classical model: $\beta$ as imaginary time.

## Outlook

- **The quantum–classical mapping ([§7.20](imaginary-time-quantum-classical.ipynb)).** $\beta$ as imaginary time, Trotter slicing,
  and the 2D Ising appointment with Onsager; then PIMC sampling the same physics ([§7.21](path-integral-monte-carlo.ipynb)).
- **Experimental chains.** CoNb₂O₆ neutron spectroscopy, Rydberg-atom arrays, trapped
  ions (outward, named).
- **Beyond ED.** DMRG and tensor networks, whose existence the area law this notebook measured explains (outward; the course's own $S$-saturation as the evidence).
- **Dynamics.** Kibble–Zurek defect production across the transition (outward, named).
- **Cross-reference** [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (the classical chain, answered; the FSS training), [§7.7](bose-einstein-fermi-dirac.ipynb)/[§7.9](fermi-gas-zero-temperature.ipynb)
  (the sea inside the magnet), [§7.18](quantum-paramagnets-brillouin.ipynb) (independence's end), [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)/[§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the algebra), [§7.4](thermal-density-matrix.ipynb)
  (log-sum-exp, again).

In [ ]:
from ecp.style import footer

footer()